<a id="top"></a>
# BB84 detector comparison

**Question.** At the same held-out 5% false-alarm target, how do pooled QBER,
basis-grouped `U2`, and state-conditioned `U4` respond to partial
intercept-resend?

Run the notebook top to bottom. The default `quick` preset finishes promptly and
produces a figure for each alarm schedule. Both figures include the held-out
noise-only point at `f=0` and the false-alarm target. Set `PRESET = "broad"`
to regenerate the full proposal grid.

This is a standalone, already-sifted BB84 detector study. It is not the QFT
repeated-secret experiment, a hardware model, or a QKD security proof.

Outputs are `bb84_sweep_results`, `bb84_calibration_summary`, and
`bb84_cumulative_results`.

[Glossary of settings and result columns](validation/MODEL_NOTES.md#glossary)


<a id="configuration"></a>
## Configuration

The first setting controls experiment breadth; the remaining settings choose
which fitted-baseline scenario appears in the figures.


In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display

from quantum_study.bb84 import (
    bb84_config,
    plot_alarm_curves,
    run_cumulative,
    run_terminal,
    summarize_calibration,
)

PRESET = "quick"  # "quick" for Run All; "broad" for the complete proposal grid.
VIEW_NOISE_PROFILE = "Erven reference"
VIEW_CALIBRATION_SIZE = 1_000  # Use 0 for the known-baseline control.
VIEW_BASELINE_FIT = 1
VIEW_EVE_P_Z = 0.5

config = bb84_config(PRESET)
print(
    f"Preset: {PRESET} · profiles={len(config.noise_profiles)}"
    f" · fitted baselines/profile={len(config.calibration_sizes) * config.baseline_fits}"
    f" · trials with one alarm decision={config.evaluation_trials:,}"
    f" · trials with scheduled alarm decisions={config.cumulative_evaluation_trials:,}"
)


<a id="methods"></a>
## Methods retained from the proposal

- Trials contain 20, 50, or 100 already-sifted checks shared by every detector.
- Pooled QBER ignores state; `U2` groups Z/X states; `U4` uses four fitted state
  rates and the proposal's score weights.
- Known-rate controls and independently fitted baselines use separate fitting,
  cutoff, null-evaluation, and attack-evaluation streams.
- The final-only test includes a large-sample normal comparison. The scheduled
  test makes decisions at 25%, 50%, and 100% of the check budget. Its cutoff
  uses the largest score across those decision points during calibration.
- Wilson intervals show Monte Carlo uncertainty. They are not finite-key
  security bounds.

The quick preset uses the Erven reference profile, uniform Eve basis choice,
six interception fractions, the true-rate control, and two independent
1,000-trial/state fits. The broad preset restores all 11 noise profiles, three
Eve biases, both calibration sizes, four fits, and the original trial counts.


<a id="terminal"></a>
## 1. Final alarm decision only

Bob decides whether to raise an alarm after collecting the full check budget. Solid
curves are held-out Monte Carlo rates, dashed curves are the analytic
single-look comparison, and bands are 95% Wilson intervals.


In [ ]:
bb84_sweep_results = run_terminal(config)
bb84_calibration_summary = summarize_calibration(bb84_sweep_results)
terminal_figure, terminal_false_alarms = plot_alarm_curves(
    bb84_sweep_results,
    config,
    profile=VIEW_NOISE_PROFILE,
    calibration_size=VIEW_CALIBRATION_SIZE,
    fit=VIEW_BASELINE_FIT,
    eve_p_z=VIEW_EVE_P_Z,
)
display(terminal_false_alarms.round(4))
plt.show()


<a id="calibration"></a>
## 2. Baseline-fit sensitivity

The compact table below shows held-out null variation across independent fitted
baselines. Power comparisons should be read only after checking these rates.


In [ ]:
calibration_null = bb84_calibration_summary[
    ~bb84_calibration_summary["is_attack"]
    & bb84_calibration_summary["noise_profile"].eq(VIEW_NOISE_PROFILE)
]
display(calibration_null[[
    "retained_checks", "calibration_trials_per_state", "detector",
    "mean_alarm_rate", "between_fit_std",
    "minimum_alarm_rate", "maximum_alarm_rate",
]].round(4))


<a id="cumulative"></a>
## 3. Multiple scheduled alarm decisions

Bob has up to three chances to raise an alarm. With a budget of 100 checks,
he decides after 25, 50, and 100 checks. Each decision uses all checks collected
so far. The 5% false-alarm target covers all three decisions in the trial.


In [ ]:
bb84_cumulative_results = run_cumulative(config)
cumulative_figure, cumulative_false_alarms = plot_alarm_curves(
    bb84_cumulative_results,
    config,
    profile=VIEW_NOISE_PROFILE,
    calibration_size=VIEW_CALIBRATION_SIZE,
    fit=VIEW_BASELINE_FIT,
    eve_p_z=VIEW_EVE_P_Z,
    cumulative=True,
)
display(cumulative_false_alarms.round(4))
plt.show()


## Reading the figures

- At `f=0`, compare each point with the black 5% line before comparing power.
- The 80% line marks a conventional power target; sampled fractions are not an
  interpolated detection threshold.
- `U2` and true-rate `U4` coincide because the configured true rates are equal
  within each basis. Fitted `U4` can differ through state-rate estimation noise.
- The broad preset is for final regeneration, not routine Run All review.


<a id="references"></a>
## Source boundaries

- Bennett and Brassard (1984), pp. 175–177, define four-state BB84, basis
  sifting, and public checking. [arXiv:2003.06557](https://arxiv.org/abs/2003.06557)
- Scarani et al. (2009), §I.B.5, give `Q=f/4` for uniform partial
  intercept-resend. [arXiv:0802.4155](https://arxiv.org/abs/0802.4155)
- Lee, Lee, and Lee (2023), §III.A, Eqs. (1)–(12), motivate Bernoulli QBER and
  normal-tail detection. [Open paper](https://dl.ifip.org/db/conf/ondm/ondm2023/1570879964.pdf)
- Lo, Chau, and Ardehali (2005), §4.2, motivate separate basis error estimates;
  `U2/U4` are project statistics. [arXiv:quant-ph/0011056](https://arxiv.org/abs/quant-ph/0011056)
- Erven et al. (2009), §4, report the `(0.002, 0.038)` reference basis rates;
  the other profiles are synthetic. [arXiv:0901.0960](https://arxiv.org/abs/0901.0960)
- Mannalath, Zapatero, and Curty (2025) motivate explicit finite-statistical
  control; their sharp bounds are not implemented here. [arXiv:2410.04095](https://arxiv.org/abs/2410.04095)
- Wilson (1927), pp. 209–212, supplies the displayed binomial intervals.
  [doi:10.1080/01621459.1927.10502953](https://doi.org/10.1080/01621459.1927.10502953)

[Back to top](#top)
